# 00. Токенизация названий и аннотаций

Первый опыт **без обучения моделей и без тематических меток**: насколько длинными становятся одинаковые тексты после токенизации ruBERT и ruSciBERT? Какая доля не помещается в 128, 256 и 512 токенов?

Вход — фиксированный локальный снимок кандидатов ОИЯИ от 10.10.2026, а не постоянно обновляющиеся реестры. Это ещё **не проверенный физический корпус**: тематический и языковой отбор не завершён. Выводы относятся только к этому массиву; опыт не проверяет H1–H3 и не сравнивает perplexity.

Веса моделей не загружаются. Реестры, исходники и допуск к обучению не меняются. Сеть по умолчанию выключена. Выбери ядро `.venv/bin/python` и выполни ячейки сверху вниз. Зависимости: `requirements-notebooks.txt`.


In [ ]:
"""Описательный опыт с двумя зафиксированными токенизаторами."""

from __future__ import annotations

import hashlib
import json
import platform
import unicodedata

from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from huggingface_hub import snapshot_download
from huggingface_hub.constants import HF_HUB_CACHE
from IPython.display import display
from transformers import AutoTokenizer, PreTrainedTokenizerBase, PreTrainedTokenizerFast

ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "src/collect").is_dir() and (directory / "README.md").is_file()),
    None,
)

if ROOT is None:
    raise RuntimeError("Открой ноутбук из проекта или каталога notebooks/.")

INPUT_PATH = ROOT / "manifests/results/h2_candidates_jinr_clean_20261010_v2/cleaned_candidates.jsonl"
EXPECTED_SHA256 = "ba5eab6ff76a34e02a50e245d095b565362d7ad1289a59b1abe74d98a92d2ae7"
OUTPUT_ROOT = ROOT / "artifacts/experiments/tokenization_pilot"
ALLOW_TOKENIZER_DOWNLOAD = False
LIMITS = (128, 256, 512)
VARIANTS = {"title": "Название", "abstract": "Аннотация", "title_abstract": "Название + аннотация"}
MODEL_SPECS = {
    "ruBERT": {
        "repo_id": "ai-forever/ruBert-base",
        "cache_alias": "ai-forever/ruBERT-base",
        "revision": "05f37a2ca9e333fd18f30cd0c96c68d274793c69",
        "required_files": ("config.json", "vocab.txt"),
        "model_type": "bert",
    },
    "ruSciBERT": {
        "repo_id": "ai-forever/ruSciBERT",
        "cache_alias": "ai-forever/ruSciBERT",
        "revision": "a721b6c9fdfe83582defdd74f9ad9afb667fb474",
        "required_files": ("config.json", "vocab.json", "merges.txt", "special_tokens_map.json"),
        "model_type": "roberta",
    },
}
TOKENIZER_FILES = (
    "config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
    "added_tokens.json", "vocab.txt", "vocab.json", "merges.txt",
)


## 1. Фиксируем вход

Берём только карточки с названием и аннотацией, без неоднозначного выбора версии этих полей. Убираем точные совпадения пары «название + аннотация» после NFKC, приведения к нижнему регистру и нормализации пробелов **только для поиска повторов**. В токенизаторы передаём исходные очищенные поля, не эту нормализованную строку.

Другие замечания сохраняем и показываем: данный фильтр не заменяет ручную проверку языка, тематики или близких версий. Все исключения записываются отдельно; случайной подвыборки нет.


In [ ]:
def _comparison_key(title: str, abstract: str) -> str:
    """Получить ключ пары полей только для поиска точных повторов."""

    fields = (title, abstract)

    return json.dumps([
        " ".join(unicodedata.normalize("NFKC", field).casefold().split())
        for field in fields
    ], ensure_ascii=False)


def prepare_inputs(records: list[dict[str, Any]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Отобрать однозначные непустые пары и явно перечислить исключения."""

    if not records:
        raise ValueError("Снимок кандидатов пуст.")

    identifiers = [record["candidate_id"] for record in records]

    if len(identifiers) != len(set(identifiers)):
        raise ValueError("В снимке повторяется candidate_id.")

    selected: list[dict[str, str]] = []
    excluded: list[dict[str, str]] = []
    seen_pairs: set[str] = set()
    ambiguous = {"multiple_title_variants", "multiple_abstract_variants"}

    for record in sorted(records, key=lambda item: item["candidate_id"]):
        title = record.get("title_clean")
        abstract = record.get("abstract_clean")
        reason = ""

        if not isinstance(title, str) or not title.strip():
            reason = "missing_title"

        elif not isinstance(abstract, str) or not abstract.strip():
            reason = "missing_abstract"

        elif ambiguous.intersection(record.get("source_review_flags", [])):
            reason = "ambiguous_text"

        if reason:
            excluded.append({"candidate_id": record["candidate_id"], "reason": reason})
            continue

        # Типы подтверждены выше; проверка также понятна анализатору типов.
        if not isinstance(title, str) or not isinstance(abstract, str):
            raise TypeError("Название и аннотация должны быть строками.")

        pair_key = _comparison_key(title, abstract)

        if pair_key in seen_pairs:
            excluded.append({"candidate_id": record["candidate_id"], "reason": "duplicate_pair"})
            continue

        seen_pairs.add(pair_key)
        selected.append({
            "candidate_id": record["candidate_id"],
            "title": title.strip(),
            "abstract": abstract.strip(),
            "title_abstract": title.strip() + "\n" + abstract.strip(),
        })

    if not selected:
        raise ValueError("После отбора не осталось текстов.")

    return pd.DataFrame(selected), pd.DataFrame(excluded, columns=["candidate_id", "reason"])


input_bytes = INPUT_PATH.read_bytes()
input_sha256 = hashlib.sha256(input_bytes).hexdigest()

if input_sha256 != EXPECTED_SHA256:
    raise ValueError("Снимок изменился. Нужна отдельная версия опыта, а не тихая подмена данных.")

records = [json.loads(line) for line in input_bytes.decode("utf-8").splitlines()]
dataset, exclusions = prepare_inputs(records)
selected_ids = set(dataset["candidate_id"])
remaining_flags = Counter(
    flag for record in records if record["candidate_id"] in selected_ids
    for flag in record.get("source_review_flags", [])
)

print(f"В снимке: {len(records)}; в опыте: {len(dataset)}; исключено: {len(exclusions)}.")
print("Причины исключения:", exclusions["reason"].value_counts().to_dict())
print("Оставшиеся отметки для проверки:", dict(sorted(remaining_flags.items())))


## 2. Загружаем только токенизаторы

Версии закреплены SHA коммитов из `docs/DECISIONS.md`. Сначала ищем файлы в локальном кеше Hugging Face. Если их нет, ячейка остановится: при необходимости можно явно разрешить скачивание **только файлов токенизатора** переключателем выше.

Сохраняем контрольные суммы файлов и конфигурации токенизации. Одинаковый размер словаря сам по себе ещё не доказывает одинаковую токенизацию. Если конфигурации совпадут, отдельной содержательной разницы ждать не следует.

У этих старых снимков нет полного `tokenizer_config.json`; результат зависит также от версии Transformers и её настроек по умолчанию. Поэтому версия библиотеки, эффективная конфигурация и контрольные примеры входят в результаты.


In [ ]:
def load_tokenizer(specification: dict[str, Any]) -> tuple[PreTrainedTokenizerBase, dict[str, Any]]:
    """Загрузить закреплённый токенизатор, не запрашивая веса модели."""

    snapshot: Path | None = None

    for repository_id in dict.fromkeys((specification["repo_id"], specification["cache_alias"])):
        cache_name = "models--" + repository_id.replace("/", "--")
        candidate = Path(HF_HUB_CACHE) / cache_name / "snapshots" / specification["revision"]

        if all((candidate / name).is_file() for name in specification["required_files"]):
            snapshot = candidate
            break

    if snapshot is None:
        if not ALLOW_TOKENIZER_DOWNLOAD:
            raise FileNotFoundError(
                f"Нет локального токенизатора {specification['repo_id']}. "
                "Для получения его файлов явно включи ALLOW_TOKENIZER_DOWNLOAD."
            )

        snapshot = Path(snapshot_download(
            repo_id=specification["repo_id"], revision=specification["revision"],
            allow_patterns=list(TOKENIZER_FILES),
        ))

    configuration = json.loads((snapshot / "config.json").read_text(encoding="utf-8"))

    if configuration.get("model_type") != specification["model_type"]:
        raise ValueError("Конфигурация не соответствует ожидаемому семейству модели.")

    # В Transformers 5.5.4 отсутствие transformers_version в старом config.json
    # может ошибочно включить проверку Mistral. Здесь явно проверены BERT/RoBERTa:
    # не применяем исправление чужого алгоритма токенизации.
    tokenizer = AutoTokenizer.from_pretrained(
        snapshot, local_files_only=True, trust_remote_code=False,
        fix_mistral_regex=False,
    )

    if not isinstance(tokenizer, PreTrainedTokenizerFast):
        raise ValueError("Этот опыт фиксирует реализацию с backend_tokenizer.")

    backend = json.loads(tokenizer.backend_tokenizer.to_str())
    canonical_backend = json.dumps(backend, sort_keys=True, ensure_ascii=False).encode("utf-8")
    file_hashes = {
        name: hashlib.sha256((snapshot / name).read_bytes()).hexdigest()
        for name in TOKENIZER_FILES if (snapshot / name).is_file()
    }
    metadata = {
        "repo_id": specification["repo_id"],
        "revision": specification["revision"],
        "model_type": configuration["model_type"],
        "tokenizer_class": type(tokenizer).__name__,
        "vocabulary_size": len(tokenizer),
        "special_tokens": tokenizer.special_tokens_map,
        "special_token_count": tokenizer.num_special_tokens_to_add(pair=False),
        "file_sha256": file_hashes,
        "backend_sha256": hashlib.sha256(canonical_backend).hexdigest(),
        "effective_backend": backend,
        "fix_mistral_regex": False,
    }

    return tokenizer, metadata


tokenizers: dict[str, PreTrainedTokenizerBase] = {}
tokenizer_metadata: dict[str, dict[str, Any]] = {}

for model_name, specification in MODEL_SPECS.items():
    tokenizers[model_name], tokenizer_metadata[model_name] = load_tokenizer(specification)

display(pd.DataFrame([
    {"model": name, **{field: metadata[field] for field in (
        "tokenizer_class", "vocabulary_size", "special_token_count", "backend_sha256",
    )}}
    for name, metadata in tokenizer_metadata.items()
]))

identical_backends = len({metadata["backend_sha256"] for metadata in tokenizer_metadata.values()}) == 1
print("Конфигурации токенизации совпадают:", identical_backends)


## 3. Длина и обрезание

Одна последовательность — название, аннотация или их объединение через перевод строки; это **не** режим пары `text_pair`. Длина включает служебные токены. Тексты не обрезаем и не дополняем до одинаковой длины.

- `median_tokens`, `p95_tokens`: медиана и 95-й процентиль длины.
- `over_limit_pct`: процент карточек, которые пришлось бы обрезать при данном ограничении.
- `lost_content_tokens_pct`: доля потерянных содержательных токенов **во всём массиве**, если каждому тексту оставить первые `limit − special_token_count` токенов.

Это модельная оценка обрезания, а не потеря смысла и не результат классификации. Ограничения 128/256/512 заданы явно: не используем огромную заглушку `model_max_length` из старых токенизаторов.


In [ ]:
def measure_lengths(
    inputs: pd.DataFrame, available_tokenizers: dict[str, PreTrainedTokenizerBase],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Посчитать длины и возможные потери на одних и тех же текстах."""

    if inputs.empty:
        raise ValueError("Нет текстов для измерения.")

    length_rows: list[dict[str, Any]] = []
    summary_rows: list[dict[str, Any]] = []
    truncation_rows: list[dict[str, Any]] = []

    for model_name, tokenizer in available_tokenizers.items():
        special_count = tokenizer.num_special_tokens_to_add(pair=False)

        for variant in VARIANTS:
            texts = inputs[variant].tolist()
            lengths: list[int] = []

            for batch_start in range(0, len(texts), 128):
                encoded = tokenizer(
                    texts[batch_start:batch_start + 128],
                    add_special_tokens=True, truncation=False, padding=False,
                    return_attention_mask=False, return_token_type_ids=False,
                    verbose=False,
                )
                batch_ids = encoded.get("input_ids")

                if not isinstance(batch_ids, list) or any(not isinstance(ids, list) for ids in batch_ids):
                    raise TypeError("Ожидался список списков идентификаторов токенов.")

                if len(batch_ids) != len(texts[batch_start:batch_start + 128]):
                    raise ValueError("Число результатов токенизации не совпало с числом текстов.")

                lengths.extend(len(token_ids) for token_ids in batch_ids)

            values = np.asarray(lengths, dtype=np.int64)
            content_tokens = values - special_count

            if np.any(content_tokens < 0):
                raise ValueError("Длина оказалась меньше числа служебных токенов.")

            for candidate_id, token_count in zip(inputs["candidate_id"], lengths, strict=True):
                length_rows.append({
                    "candidate_id": candidate_id, "model": model_name,
                    "input": variant, "tokens": token_count,
                })

            summary_rows.append({
                "model": model_name, "input": variant, "documents": len(values),
                "median_tokens": float(np.median(values)),
                "p95_tokens": float(np.quantile(values, 0.95)),
                "max_tokens": int(values.max()),
                "total_content_tokens": int(content_tokens.sum()),
            })

            for limit in LIMITS:
                if limit <= special_count:
                    raise ValueError("Ограничение должно превышать число служебных токенов.")

                lost_tokens = np.maximum(values - limit, 0).sum()
                truncation_rows.append({
                    "model": model_name, "input": variant, "limit": limit,
                    "over_limit_pct": float(100 * np.mean(values > limit)),
                    "lost_content_tokens_pct": float(
                        100 * lost_tokens / max(int(content_tokens.sum()), 1)
                    ),
                })

    return pd.DataFrame(length_rows), pd.DataFrame(summary_rows), pd.DataFrame(truncation_rows)


lengths, length_summary, truncation_summary = measure_lengths(dataset, tokenizers)
display(length_summary.round(2))
display(truncation_summary.round(2))

title_lengths = lengths.loc[lengths["input"] == "title"].pivot(
    index="candidate_id", columns="model", values="tokens",
)
title_lengths["all_upper"] = dataset.set_index("candidate_id")["title"].map(str.isupper)
casing_summary = title_lengths.groupby("all_upper").agg(
    documents=("ruBERT", "size"),
    rubert_median=("ruBERT", "median"),
    ruscibert_median=("ruSciBERT", "median"),
).reset_index()
print("Длина названий отдельно для сплошных заглавных букв (all_upper):")
display(casing_summary)

figure, axis = plt.subplots(figsize=(9, 4))

for model_name in tokenizers:
    values = np.sort(lengths.loc[
        (lengths["model"] == model_name) & (lengths["input"] == "title_abstract"), "tokens",
    ].to_numpy())
    axis.step(values, np.arange(1, len(values) + 1) / len(values), label=model_name)

for limit in LIMITS:
    axis.axvline(limit, linestyle=":", color="gray", alpha=0.6)

axis.set(
    xlabel="Токенов в названии + аннотации, включая служебные",
    ylabel="Накопленная доля карточек",
    title="Распределение длин: одинаковый вход, разные токенизаторы",
)
axis.legend()
figure.tight_layout()
plt.show()


## 4. Несколько физических терминов

В таблице выше отдельно проверяется регистр названий: `all_upper=True`, если все буквы, имеющие регистр, заглавные. У закреплённого ruBERT нормализатор приводит текст к нижнему регистру; у ruSciBERT этот шаг отсутствует. Разница длин может быть связана и с оформлением метаданных. Это наблюдение, не контролируемая проверка эффекта регистра: для неё нужен отдельный парный опыт. Основные тексты здесь не переводим в нижний регистр.

Далее — заранее заданные примеры, а не репрезентативный тест терминологии. Считаем токены без служебных знаков. Для ruSciBERT внутренние строковые обозначения могут выглядеть непривычно: его байтовая BPE-запись не является испорченным русским текстом.

Более короткая последовательность удобнее для бюджета контекста, но не доказывает, что модель лучше понимает термин.


In [ ]:
TERMS = (
    "электрон", "фотон", "сверхпроводимость", "наноструктура", "ферромагнетизм",
    "квантовая электродинамика", "плазменная неустойчивость", "рассеяние нейтронов",
)
term_rows: list[dict[str, Any]] = []

for model_name, tokenizer in tokenizers.items():
    for term in TERMS:
        token_ids = tokenizer.encode(term, add_special_tokens=False)
        term_rows.append({
            "model": model_name, "term": term, "tokens": len(token_ids),
            "pieces": tokenizer.convert_ids_to_tokens(token_ids),
            "token_ids": token_ids,
        })

terms = pd.DataFrame(term_rows)
display(terms.pivot(index="term", columns="model", values="tokens").reindex(TERMS))


## 5. Сохраняем воспроизводимый результат

Результаты и перечень выбранных карточек пишутся в уже игнорируемый Git каталог `artifacts/experiments/`, отдельная папка на каждый запуск. Параметры и SHA-256 фиксируются в `run.json`. Исходные тексты туда не копируем: вход закреплён контрольной суммой.

Сохраняем версии пакетов, Python и хеш кода ячеек. Перед коммитом самого ноутбука очищаем его выводы. На другой машине для повторения потребуется тот же локальный снимок и файлы токенизаторов.


In [ ]:
run_time = datetime.now(timezone.utc)
output_dir = OUTPUT_ROOT / run_time.strftime("%Y%m%dT%H%M%S_%fZ")
output_dir.mkdir(parents=True, exist_ok=False)

for name, frame in {
    "lengths": lengths,
    "length_summary": length_summary,
    "truncation_summary": truncation_summary,
    "exclusions": exclusions,
    "title_casing_summary": casing_summary,
    "selected_candidates": dataset[["candidate_id"]],
}.items():
    frame.to_csv(output_dir / f"{name}.csv", index=False)

(output_dir / "term_examples.json").write_text(
    json.dumps(term_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)
figure.savefig(output_dir / "lengths.png", dpi=150, bbox_inches="tight")

notebook_path = ROOT / "notebooks/00_tokenization_pilot.ipynb"
notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
code_cells = ["".join(cell["source"]) for cell in notebook["cells"] if cell["cell_type"] == "code"]
code_sha256 = hashlib.sha256(json.dumps(code_cells, ensure_ascii=False).encode("utf-8")).hexdigest()
run_metadata = {
    "experiment_kind": "descriptive_tokenization_pilot",
    "formal_hypothesis_result": False,
    "created_at": run_time.isoformat(),
    "input_path": str(INPUT_PATH.relative_to(ROOT)),
    "input_sha256": input_sha256,
    "notebook_code_sha256": code_sha256,
    "python_version": platform.python_version(),
    "versions": {name: version(name) for name in (
        "transformers", "tokenizers", "huggingface-hub", "numpy", "pandas", "matplotlib",
    )},
    "selection": {
        "input_count": len(records), "selected_count": len(dataset),
        "excluded_by_reason": exclusions["reason"].value_counts().to_dict(),
        "remaining_review_flags": dict(remaining_flags),
        "deduplication": "NFKC + casefold + whitespace; pair of title and abstract",
        "sampling": "all eligible input pairs, no random subsampling",
    },
    "variants": VARIANTS,
    "limits_including_special_tokens": LIMITS,
    "tokenizers": tokenizer_metadata,
    "identical_backends": identical_backends,
}
(output_dir / "run.json").write_text(
    json.dumps(run_metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)
print(f"Результаты сохранены: {output_dir.relative_to(ROOT)}")


## Как читать результат

На этом этапе выбираем разумную длину последовательности для следующего опыта и видим цену обрезания. **Победителя по пониманию физики здесь нет.** Такой вывод требует размеченной классификации; H1 требует отдельного протокола MLM.

Дальше: `01_classification_baseline.ipynb`. Для реального запуска ему нужны проверенные метки и разделение по группам связанных статей. Источники описания инструментов: [токенизаторы Transformers](https://huggingface.co/docs/transformers/main_classes/tokenizer) и [закреплённые модели проекта](../docs/DECISIONS.md).
